# 🔥 Chapter 6 — Building Neural Networks with `nn.Module`

---

### 🎯 What you'll learn
- `nn.Module` — the base class for all models
- Common layers: Linear, Conv2d, RNN, LSTM
- Activation functions and when to use each
- Model inspection, parameter counting
- ⚠️ Corner cases: weight initialization, forgetting `super().__init__()`

## 6.1 — Your First Neural Network

In [ ]:
import torch
import torch.nn as nn

class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()  # ⚠️ NEVER forget this!
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = SimpleNet(input_size=10, hidden_size=32, output_size=3)
print(model)

# Test with random input
x = torch.randn(5, 10)  # batch of 5, 10 features each
output = model(x)        # Calls forward() internally
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {output.shape}")

## 6.2 — ⚠️ Corner Case: NEVER Call `forward()` Directly!

In [ ]:
# ❌ DON'T: model.forward(x)
# ✅ DO:    model(x)

# Why? model(x) calls __call__ which:
# 1. Runs all registered hooks (for debugging, logging)
# 2. Handles gradient tracking
# 3. THEN calls forward()

# model.forward(x) skips hooks and can cause subtle bugs!
print("Always use model(x), never model.forward(x)!")

## 6.3 — Using `nn.Sequential` for Simple Models

In [ ]:
# When the model is a simple stack of layers, use Sequential
model_seq = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Dropout(0.2),       # Randomly zero 20% of inputs (regularization)
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
)

print(model_seq)
output = model_seq(torch.randn(5, 10))
print(f"Output shape: {output.shape}")

## 6.4 — Common Layers Reference

In [ ]:
# ── Fully Connected (Linear) ──
linear = nn.Linear(in_features=128, out_features=64, bias=True)
print(f"Linear weight shape: {linear.weight.shape}")  # (64, 128)

# ── Convolutional (2D) ──
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
img = torch.randn(1, 3, 32, 32)  # (batch, channels, height, width)
print(f"Conv2d input:  {img.shape}")
print(f"Conv2d output: {conv(img).shape}")  # (1, 16, 32, 32) — same spatial size due to padding=1

# ── Batch Normalization ──
bn = nn.BatchNorm2d(num_features=16)  # Match channels of previous layer

# ── Pooling ──
pool = nn.MaxPool2d(kernel_size=2, stride=2)
print(f"After pool: {pool(conv(img)).shape}")  # (1, 16, 16, 16) — halved

# ── Recurrent (LSTM) ──
lstm = nn.LSTM(input_size=50, hidden_size=100, num_layers=2, batch_first=True)
seq = torch.randn(8, 20, 50)  # (batch, seq_len, features)
output, (hidden, cell) = lstm(seq)
print(f"LSTM output: {output.shape}")  # (8, 20, 100)

## 6.5 — Activation Functions: When to Use What

In [ ]:
import torch.nn.functional as F

x = torch.linspace(-5, 5, 100)

# ReLU — default choice for hidden layers
# ⚠️ Corner Case: "Dying ReLU" — neurons can permanently output 0
relu = torch.relu(x)

# LeakyReLU — fixes dying ReLU with small negative slope
leaky = F.leaky_relu(x, negative_slope=0.01)

# GELU — used in Transformers (GPT, BERT)
gelu = F.gelu(x)

# Sigmoid — output layer for binary classification (range 0-1)
# ⚠️ Corner Case: saturates at extremes → vanishing gradients
sigmoid = torch.sigmoid(x)

# Softmax — output layer for multi-class classification (sums to 1)
# ⚠️ Corner Case: use dim= parameter correctly!
logits = torch.randn(3, 5)
probs = F.softmax(logits, dim=1)   # Softmax over classes (dim=1), NOT batch (dim=0)!
print(f"Probs sum per sample: {probs.sum(dim=1)}")  # Should be [1, 1, 1]

## 6.6 — Model Inspection & Parameter Counting

In [ ]:
model = SimpleNet(784, 256, 10)

# List all parameters
for name, param in model.named_parameters():
    print(f"{name:20s} shape={str(param.shape):15s} requires_grad={param.requires_grad}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6.7 — ⚠️ Weight Initialization Corner Cases

In [ ]:
model = SimpleNet(784, 256, 10)

# ⚠️ Default init may not be optimal for deep networks
# Apply custom initialization
def init_weights(m):
    if isinstance(m, nn.Linear):
        # Kaiming (He) init — best for ReLU networks
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)

model.apply(init_weights)
print("Kaiming initialization applied!")

# Other common initializations:
# nn.init.xavier_uniform_(m.weight)   # Best for Sigmoid/Tanh
# nn.init.normal_(m.weight, mean=0, std=0.01)
# nn.init.constant_(m.bias, 0)

## ✏️ Exercises

In [ ]:
# Exercise 1: Build a CNN for 28x28 grayscale images (like MNIST)
# Architecture: Conv→ReLU→Pool→Conv→ReLU→Pool→Flatten→FC→FC
# Print the model and verify output shape with a dummy input.
# YOUR CODE HERE


# Exercise 2: Write a function that prints a table of all layer names,
# output shapes (given an input), and parameter counts.
# YOUR CODE HERE
